# Exercise 2.3: Comparing float16, float32 & float64 Precision

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vuhung16au/Maths-for-IT-Notebooks/blob/main/Chapter-02/03-float16-vs-float32-precision.ipynb)

**Book Reference:** Chapter 2 — Floating-point Range and Precision (Section 2.3)  
**Estimated Time:** 15–20 min

### Learning Objectives
- Measure machine epsilon (unit roundoff) empirically for `np.float16`, `np.float32`, and `np.float64`.
- Observe overflow to `inf` in `float16` (max ~65504) vs `float32`.
- Quantify cumulative rounding error when summing arrays in lower precision.


In [ ]:
import warnings
import numpy as np

rng = np.random.default_rng(42)


### Mathematical Context

Modern AI training and inference frequently use half-precision (`float16` or `bfloat16`) to save memory and accelerate matrix multiplications. However, `np.float16` has only 10 significand bits (machine epsilon $\approx 2^{-10} \approx 9.77 \times 10^{-4}$) and a maximum finite value of `65504`.
In contrast, `np.float32` has 23 significand bits ($\epsilon \approx 1.19 \times 10^{-7}$), and `np.float64` has 52 significand bits ($\epsilon \approx 2.22 \times 10^{-16}$).


In [ ]:
def measure_machine_epsilon(dtype: type) -> float:
    """Empirically find machine epsilon: smallest 2^(-k) such that dtype(1.0) + dtype(eps) != dtype(1.0)."""
    # TODO: Start with eps = 1.0 and halve it while dtype(1.0) + dtype(eps / 2.0) != dtype(1.0).
    raise NotImplementedError("Complete this TODO")


def summation_error(values: np.ndarray, dtype: type) -> float:
    """Sum `values` cast to `dtype` and return absolute error vs float64 sum."""
    # TODO: Compute float64 reference sum and dtype sum, then return abs(ref_sum - low_sum).
    raise NotImplementedError("Complete this TODO")


def overflows_to_inf(x: float, dtype: type) -> bool:
    """Return True if converting x to `dtype` overflows to infinity."""
    # TODO: Cast x to dtype (suppressing RuntimeWarning if needed) and check np.isinf.
    raise NotImplementedError("Complete this TODO")


<details><summary>Hint</summary>

- Be sure to cast both `1.0` and `eps / 2.0` to `dtype` before adding, otherwise Python promotes to `float64`!
- In NumPy 2.x, `dtype(70000.0)` for `np.float16` may emit a `RuntimeWarning`; `warnings.catch_warnings()` keeps output clean.

</details>


In [ ]:
for dt in (np.float16, np.float32, np.float64):
    eps = measure_machine_epsilon(dt)
    expected = float(np.finfo(dt).eps)
    print(f'{dt.__name__:8s} | measured eps={eps:.3e} | finfo eps={expected:.3e}')
    assert np.isclose(eps, expected)

assert overflows_to_inf(70000.0, np.float16) is True
assert overflows_to_inf(70000.0, np.float32) is False

data = rng.uniform(0.0, 1.0, size=2000)
err16 = summation_error(data, np.float16)
err32 = summation_error(data, np.float32)
print(f'Summation error (N=2000): float16={err16:.4e}, float32={err32:.4e}')
assert err16 > err32
print('All Chapter 2.3 checks passed!')


### Think Deeper

1. Why does mixed-precision training accumulate gradients in `float32` even when forward/backward passes use `float16`?
2. What happens if a neural network loss or exponent exceeds `65504` in `float16`?
